In [10]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ============================================================
# Config
# ============================================================
CSV_PATH = "mimic_results.csv"   # <- change if needed
OUT_DIR = "taskwise_results_mimic"
os.makedirs(OUT_DIR, exist_ok=True)
AUG_CSV_PATH = "mimic_aug_results.csv"   # optional
INCLUDE_AUG_TARGET = True
AUG_TARGET_NAME = "eICU-Augmented"

SOURCE_NAME = "MIMIC"   # <- change if needed
TARGET_NAME = "eICU"    # <- change if needed

FIGSIZE = (12, 5.8)
DPI = 180

PALETTE = [
    "#8FB9A8",  # muted teal
    "#F2C6A0",  # soft peach
    "#A8C3D1",  # desaturated blue
    "#D6A2AD",  # dusty rose
]


# ============================================================
# Pretty display names
# ============================================================
DISPLAY_NAME_MAP = {
    "Transformer": "Transformer",
    "Transmean": "Transmean",
    "Raindrop": "Raindrop",
    "ViTST_swin": "ViTST-Swin",
    "ViTST_vit": "ViTST-ViT",
    "strats": "STraTS",
    "surprise": "Surprise",
    "surprise_vt": "SurpVT",
    "surprise_vttg": "SurpVTTG",
}

METHOD_ORDER = [
    "Transformer",
    "Transmean",
    "Raindrop",
    "ViTST_swin",
    "ViTST_vit",
    "strats",
    "surprise",
    "surprise_vt",
    "surprise_vttg",
]


# ============================================================
# Helpers
# ============================================================
def parse_base_model(name: str) -> str:
    s = str(name)

    # Case 1: simple names like ViTST_vit_mimic_5 / surprise_vttg_eicu_3
    m = re.match(r"^(.*)_(mimic|eicu)_(\d+)$", s, flags=re.IGNORECASE)
    if m:
        return m.group(1)

    # Case 2: long names like ViTST_vit_mimic_to_eicu_split4__srcOrig__tgtAug-aug
    m = re.match(r"^(.*)_(mimic|eicu)_to_(mimic|eicu)_split\d+.*$", s, flags=re.IGNORECASE)
    if m:
        return m.group(1)

    return s

def parse_run_id(name: str) -> str:
    s = str(name)

    # Format 1: ViTST_vit_mimic_5
    m = re.match(r"^(.*)_(mimic|eicu)_(\d+)$", s, flags=re.IGNORECASE)
    if m:
        return m.group(3)

    # Format 2: ViTST_vit_mimic_to_eicu_split4__srcOrig__tgtAug-aug
    m = re.match(r"^(.*)_(mimic|eicu)_to_(mimic|eicu)_split(\d+).*$", s, flags=re.IGNORECASE)
    if m:
        return m.group(4)

    return s

def summarize_phenotype_from_raw(
    df: pd.DataFrame,
    aug_df: pd.DataFrame = None,
    exclude_tasks=None,
    verbose=False,
):
    """
    Phenotype summary should be:
      1) for each run, average phenotype tasks -> macro performance
      2) then average macro performance across runs for each model
    """
    if exclude_tasks is None:
        exclude_tasks = {"readm_label", "los_label", "mor_label"}

    all_tasks = extract_tasks(df)
    phenotype_tasks = [t for t in all_tasks if t not in exclude_tasks]

    if len(phenotype_tasks) == 0:
        raise ValueError("No phenotype tasks found.")

    work = df.copy()
    work["base_model"] = work["Name"].apply(parse_base_model)
    work["run_id"] = work["Name"].apply(parse_run_id)

    # ------------------------------------------------------------
    # Step 1: compute per-run macro over phenotype tasks
    # ------------------------------------------------------------
    run_rows = []
    for _, row in work.iterrows():
        src_auroc_vals = []
        src_auprc_vals = []
        tgt_auroc_vals = []
        tgt_auprc_vals = []

        for task in phenotype_tasks:
            c1 = f"test/source_auroc_{task}"
            c2 = f"test/source_auprc_{task}"
            c3 = f"test/target_auroc_{task}"
            c4 = f"test/target_auprc_{task}"

            if c1 in row.index and pd.notna(row[c1]):
                src_auroc_vals.append(row[c1])
            if c2 in row.index and pd.notna(row[c2]):
                src_auprc_vals.append(row[c2])
            if c3 in row.index and pd.notna(row[c3]):
                tgt_auroc_vals.append(row[c3])
            if c4 in row.index and pd.notna(row[c4]):
                tgt_auprc_vals.append(row[c4])

        run_rows.append({
            "base_model": row["base_model"],
            "run_id": row["run_id"],
            "source_auroc_macro": np.mean(src_auroc_vals) if len(src_auroc_vals) else np.nan,
            "source_auprc_macro": np.mean(src_auprc_vals) if len(src_auprc_vals) else np.nan,
            "target_auroc_macro": np.mean(tgt_auroc_vals) if len(tgt_auroc_vals) else np.nan,
            "target_auprc_macro": np.mean(tgt_auprc_vals) if len(tgt_auprc_vals) else np.nan,
        })

    run_macro = pd.DataFrame(run_rows)

    # If the input CSV has one row per run, this is enough.
    # If duplicate rows somehow exist for same model/run, collapse them.
    run_macro = run_macro.groupby(["base_model", "run_id"], as_index=False).agg({
        "source_auroc_macro": "mean",
        "source_auprc_macro": "mean",
        "target_auroc_macro": "mean",
        "target_auprc_macro": "mean",
    })

    # ------------------------------------------------------------
    # Optional augmented target phenotype macro
    # same logic, but only target metrics come from aug_df
    # ------------------------------------------------------------
    if aug_df is not None:
        aug_work = aug_df.copy()
        aug_work["base_model"] = aug_work["Name"].apply(parse_base_model)
        aug_work["run_id"] = aug_work["Name"].apply(parse_run_id)

        aug_rows = []
        for _, row in aug_work.iterrows():
            tgt_aug_auroc_vals = []
            tgt_aug_auprc_vals = []

            for task in phenotype_tasks:
                c1 = f"test/target_auroc_{task}"
                c2 = f"test/target_auprc_{task}"

                if c1 in row.index and pd.notna(row[c1]):
                    tgt_aug_auroc_vals.append(row[c1])
                if c2 in row.index and pd.notna(row[c2]):
                    tgt_aug_auprc_vals.append(row[c2])

            aug_rows.append({
                "base_model": row["base_model"],
                "run_id": row["run_id"],
                "target_aug_auroc_macro": np.mean(tgt_aug_auroc_vals) if len(tgt_aug_auroc_vals) else np.nan,
                "target_aug_auprc_macro": np.mean(tgt_aug_auprc_vals) if len(tgt_aug_auprc_vals) else np.nan,
            })

        aug_run_macro = pd.DataFrame(aug_rows)
        aug_run_macro = aug_run_macro.groupby(["base_model", "run_id"], as_index=False).agg({
            "target_aug_auroc_macro": "mean",
            "target_aug_auprc_macro": "mean",
        })

        run_macro = run_macro.merge(
            aug_run_macro,
            on=["base_model", "run_id"],
            how="left"
        )

    if verbose:
        print("\n[phenotype] tasks used:")
        print(phenotype_tasks)
        print("\n[phenotype] per-run macro rows:")
        print(run_macro.head())

    # ------------------------------------------------------------
    # Step 2: average per-run macro over runs for each model
    # ------------------------------------------------------------
    agg_dict = {
        "source_auroc_macro": ["mean", "std"],
        "source_auprc_macro": ["mean", "std"],
        "target_auroc_macro": ["mean", "std"],
        "target_auprc_macro": ["mean", "std"],
    }

    if aug_df is not None:
        agg_dict["target_aug_auroc_macro"] = ["mean", "std"]
        agg_dict["target_aug_auprc_macro"] = ["mean", "std"]

    pheno = run_macro.groupby("base_model", as_index=False).agg(agg_dict)

    cols = [
        "base_model",
        "test/source_auroc_phenotype_mean", "test/source_auroc_phenotype_std",
        "test/source_auprc_phenotype_mean", "test/source_auprc_phenotype_std",
        "test/target_auroc_phenotype_mean", "test/target_auroc_phenotype_std",
        "test/target_auprc_phenotype_mean", "test/target_auprc_phenotype_std",
    ]

    if aug_df is not None:
        cols += [
            "target_aug_auroc_mean", "target_aug_auroc_std",
            "target_aug_auprc_mean", "target_aug_auprc_std",
        ]

    pheno.columns = cols
    pheno["display_name"] = pheno["base_model"].apply(get_display_name)
    pheno = reorder_methods(pheno)

    return pheno, phenotype_tasks, run_macro

def make_readable_table_phenotype(task_summary: pd.DataFrame, include_aug_target=False):
    out = pd.DataFrame({
        "Methods": task_summary["display_name"],
        "Source AUROC": [
            fmt_mean_std(m, s) for m, s in zip(
                task_summary["test/source_auroc_phenotype_mean"],
                task_summary["test/source_auroc_phenotype_std"]
            )
        ],
        "Source AUPRC": [
            fmt_mean_std(m, s) for m, s in zip(
                task_summary["test/source_auprc_phenotype_mean"],
                task_summary["test/source_auprc_phenotype_std"]
            )
        ],
        "Target AUROC": [
            fmt_mean_std(m, s) for m, s in zip(
                task_summary["test/target_auroc_phenotype_mean"],
                task_summary["test/target_auroc_phenotype_std"]
            )
        ],
        "Target AUPRC": [
            fmt_mean_std(m, s) for m, s in zip(
                task_summary["test/target_auprc_phenotype_mean"],
                task_summary["test/target_auprc_phenotype_std"]
            )
        ],
    })

    if include_aug_target:
        out["Target Aug AUROC"] = [
            fmt_mean_std(m, s) for m, s in zip(
                task_summary["target_aug_auroc_mean"],
                task_summary["target_aug_auroc_std"]
            )
        ]
        out["Target Aug AUPRC"] = [
            fmt_mean_std(m, s) for m, s in zip(
                task_summary["target_aug_auprc_mean"],
                task_summary["target_aug_auprc_std"]
            )
        ]

    return out

def plot_phenotype_performance(task_summary: pd.DataFrame, out_dir: str, include_aug_target=False, aug_target_name="Target-Aug"):
    models = task_summary["display_name"].tolist()
    x = np.arange(len(models))
    width = 0.16 if include_aug_target else 0.2

    s_auroc = task_summary["test/source_auroc_phenotype_mean"].to_numpy() * 100
    s_auprc = task_summary["test/source_auprc_phenotype_mean"].to_numpy() * 100
    t_auroc = task_summary["test/target_auroc_phenotype_mean"].to_numpy() * 100
    t_auprc = task_summary["test/target_auprc_phenotype_mean"].to_numpy() * 100

    s_auroc_std = task_summary["test/source_auroc_phenotype_std"].fillna(0).to_numpy() * 100
    s_auprc_std = task_summary["test/source_auprc_phenotype_std"].fillna(0).to_numpy() * 100
    t_auroc_std = task_summary["test/target_auroc_phenotype_std"].fillna(0).to_numpy() * 100
    t_auprc_std = task_summary["test/target_auprc_phenotype_std"].fillna(0).to_numpy() * 100

    fig, ax = plt.subplots(figsize=FIGSIZE)

    if include_aug_target:
        ta_auroc = task_summary["target_aug_auroc_mean"].to_numpy() * 100
        ta_auprc = task_summary["target_aug_auprc_mean"].to_numpy() * 100
        ta_auroc_std = task_summary["target_aug_auroc_std"].fillna(0).to_numpy() * 100
        ta_auprc_std = task_summary["target_aug_auprc_std"].fillna(0).to_numpy() * 100

        aug_palette = PALETTE + ["#B7B1D8", "#E7CFC4"]

        ax.bar(x - 2.5 * width, s_auroc, width, yerr=s_auroc_std, capsize=3, label=f"Source AUROC ({SOURCE_NAME})", color=aug_palette[0])
        ax.bar(x - 1.5 * width, s_auprc, width, yerr=s_auprc_std, capsize=3, label=f"Source AUPRC ({SOURCE_NAME})", color=aug_palette[1])
        ax.bar(x - 0.5 * width, t_auroc, width, yerr=t_auroc_std, capsize=3, label=f"Target AUROC ({TARGET_NAME})", color=aug_palette[2])
        ax.bar(x + 0.5 * width, t_auprc, width, yerr=t_auprc_std, capsize=3, label=f"Target AUPRC ({TARGET_NAME})", color=aug_palette[3])
        ax.bar(x + 1.5 * width, ta_auroc, width, yerr=ta_auroc_std, capsize=3, label=f"Target AUROC ({aug_target_name})", color=aug_palette[4])
        ax.bar(x + 2.5 * width, ta_auprc, width, yerr=ta_auprc_std, capsize=3, label=f"Target AUPRC ({aug_target_name})", color=aug_palette[5])
    else:
        ax.bar(x - 1.5 * width, s_auroc, width, yerr=s_auroc_std, capsize=3, label=f"Source AUROC ({SOURCE_NAME})", color=PALETTE[0])
        ax.bar(x - 0.5 * width, s_auprc, width, yerr=s_auprc_std, capsize=3, label=f"Source AUPRC ({SOURCE_NAME})", color=PALETTE[1])
        ax.bar(x + 0.5 * width, t_auroc, width, yerr=t_auroc_std, capsize=3, label=f"Target AUROC ({TARGET_NAME})", color=PALETTE[2])
        ax.bar(x + 1.5 * width, t_auprc, width, yerr=t_auprc_std, capsize=3, label=f"Target AUPRC ({TARGET_NAME})", color=PALETTE[3])

    ax.set_xticks(x)
    ax.set_xticklabels(models, rotation=35, ha="right")
    ax.set_ylabel("Score (%)")
    ax.set_title("Task-wise performance: phenotype")

    ax.legend(
        loc="upper center",
        bbox_to_anchor=(0.5, -0.18),
        ncol=3 if include_aug_target else 2,
        frameon=False
    )

    ax.grid(axis="y", linestyle="--", alpha=0.25)
    fig.tight_layout(rect=[0, 0.15, 1, 1])

    save_path = os.path.join(out_dir, "phenotype_performance.png")
    fig.savefig(save_path, dpi=DPI, bbox_inches="tight")
    plt.close(fig)

    return save_path

def make_latex_table_phenotype(
    task_summary: pd.DataFrame,
    source_name: str,
    target_name: str,
    include_aug_target=False,
    aug_target_name="Target-Aug"
):
    cols = {
        "source_auroc": "test/source_auroc_phenotype_mean",
        "source_auprc": "test/source_auprc_phenotype_mean",
        "target_auroc": "test/target_auroc_phenotype_mean",
        "target_auprc": "test/target_auprc_phenotype_mean",
    }
    std_cols = {
        "source_auroc": "test/source_auroc_phenotype_std",
        "source_auprc": "test/source_auprc_phenotype_std",
        "target_auroc": "test/target_auroc_phenotype_std",
        "target_auprc": "test/target_auprc_phenotype_std",
    }

    if include_aug_target:
        cols["target_aug_auroc"] = "target_aug_auroc_mean"
        cols["target_aug_auprc"] = "target_aug_auprc_mean"
        std_cols["target_aug_auroc"] = "target_aug_auroc_std"
        std_cols["target_aug_auprc"] = "target_aug_auprc_std"

    rank_marks = {}
    for key, mean_col in cols.items():
        rank_marks[key] = get_top2_mask(task_summary[mean_col])

    lines = []
    lines.append("\\begin{table}[!t]")
    lines.append("\\scriptsize")
    lines.append("\\centering")
    lines.append(
        f"\\caption{{Phenotype macro performance, computed by first averaging phenotype tasks "
        f"within each run and then averaging across runs, when training on {source_name} and "
        f"evaluating on {source_name} (source), {target_name} (target)"
        + (f", and {aug_target_name} (augmented target)" if include_aug_target else "")
        + ".}"
    )
    lines.append("\\label{tab:phenotype_macro}")
    lines.append("\\resizebox{\\textwidth}{!}{")

    if include_aug_target:
        lines.append("\\begin{tabular}{l|ll|ll|ll}")
        lines.append("\\toprule")
        lines.append(
            f"& \\multicolumn{{2}}{{c|}}{{Source ({source_name})}} "
            f"& \\multicolumn{{2}}{{c|}}{{Target ({target_name})}} "
            f"& \\multicolumn{{2}}{{c}}{{Target ({aug_target_name})}} \\\\"
        )
        lines.append("\\cmidrule{2-7}")
        lines.append("\\multirow{-2}{*}{Methods} & AUROC & AUPRC & AUROC & AUPRC & AUROC & AUPRC \\\\")
    else:
        lines.append("\\begin{tabular}{l|ll|ll}")
        lines.append("\\toprule")
        lines.append(
            f"& \\multicolumn{{2}}{{c|}}{{Source ({source_name})}} "
            f"& \\multicolumn{{2}}{{c}}{{Target ({target_name})}} \\\\"
        )
        lines.append("\\cmidrule{2-5}")
        lines.append("\\multirow{-2}{*}{Methods} & AUROC & AUPRC & AUROC & AUPRC \\\\")

    lines.append("\\midrule")

    for idx, row in task_summary.iterrows():
        m = row["display_name"]

        s_auroc = latex_cell(row[cols["source_auroc"]], row[std_cols["source_auroc"]], rank_marks["source_auroc"].get(idx))
        s_auprc = latex_cell(row[cols["source_auprc"]], row[std_cols["source_auprc"]], rank_marks["source_auprc"].get(idx))
        t_auroc = latex_cell(row[cols["target_auroc"]], row[std_cols["target_auroc"]], rank_marks["target_auroc"].get(idx))
        t_auprc = latex_cell(row[cols["target_auprc"]], row[std_cols["target_auprc"]], rank_marks["target_auprc"].get(idx))

        if include_aug_target:
            ta_auroc = latex_cell(row[cols["target_aug_auroc"]], row[std_cols["target_aug_auroc"]], rank_marks["target_aug_auroc"].get(idx))
            ta_auprc = latex_cell(row[cols["target_aug_auprc"]], row[std_cols["target_aug_auprc"]], rank_marks["target_aug_auprc"].get(idx))
            lines.append(f"{m} & {s_auroc} & {s_auprc} & {t_auroc} & {t_auprc} & {ta_auroc} & {ta_auprc}\\\\")
        else:
            lines.append(f"{m} & {s_auroc} & {s_auprc} & {t_auroc} & {t_auprc}\\\\")

    lines.append("\\bottomrule")
    lines.append("\\end{tabular}")
    lines.append("}")
    lines.append("\\end{table}")

    return "\n".join(lines)

def pretty_task_name(task: str) -> str:
    return task.replace("_", " ")


def get_display_name(base_model: str) -> str:
    name = DISPLAY_NAME_MAP.get(base_model, base_model)
    return name.replace("_", " ")


def fmt_mean_std(mean_val, std_val, scale100=True):
    if pd.isna(mean_val):
        return "-"
    if scale100:
        return f"{mean_val * 100:.1f}±{std_val * 100:.1f}"
    return f"{mean_val:.4f}±{std_val:.4f}"


def extract_tasks(df: pd.DataFrame):
    tasks = set()
    pat = re.compile(r"^test/(source|target)_(auroc|auprc)_(.+)$")
    for c in df.columns:
        m = pat.match(c)
        if m:
            tasks.add(m.group(3))
    return sorted(tasks)


def get_metric_columns_for_task(task, include_aug_target=False):
    cols = {
        "source_auroc": f"test/source_auroc_{task}",
        "source_auprc": f"test/source_auprc_{task}",
        "target_auroc": f"test/target_auroc_{task}",
        "target_auprc": f"test/target_auprc_{task}",
    }

    if include_aug_target:
        cols["target_aug_auroc"] = f"test/target_auroc_{task}"
        cols["target_aug_auprc"] = f"test/target_auprc_{task}"

    return cols

def reorder_methods(df_summary: pd.DataFrame):
    df_summary = df_summary.copy()
    df_summary["__order"] = df_summary["base_model"].apply(
        lambda x: METHOD_ORDER.index(x) if x in METHOD_ORDER else 999
    )
    df_summary = df_summary.sort_values(["__order", "display_name"]).drop(columns="__order")
    return df_summary.reset_index(drop=True)


def summarize_task(df: pd.DataFrame, task: str, aug_df: pd.DataFrame = None, verbose=True):
    src_auroc = f"test/source_auroc_{task}"
    src_auprc = f"test/source_auprc_{task}"
    tgt_auroc = f"test/target_auroc_{task}"
    tgt_auprc = f"test/target_auprc_{task}"

    work = df.copy()
    work["base_model"] = work["Name"].apply(parse_base_model)

    out = work.groupby("base_model", as_index=False).agg({
        src_auroc: ["mean", "std"],
        src_auprc: ["mean", "std"],
        tgt_auroc: ["mean", "std"],
        tgt_auprc: ["mean", "std"],
    })

    out.columns = [
        "base_model",
        f"{src_auroc}_mean", f"{src_auroc}_std",
        f"{src_auprc}_mean", f"{src_auprc}_std",
        f"{tgt_auroc}_mean", f"{tgt_auroc}_std",
        f"{tgt_auprc}_mean", f"{tgt_auprc}_std",
    ]

    if aug_df is not None:
        aug_work = aug_df.copy()
        aug_work["base_model"] = aug_work["Name"].apply(parse_base_model)

        aug_out = aug_work.groupby("base_model", as_index=False).agg({
            tgt_auroc: ["mean", "std"],
            tgt_auprc: ["mean", "std"],
        })

        aug_out.columns = [
            "base_model",
            "target_aug_auroc_mean", "target_aug_auroc_std",
            "target_aug_auprc_mean", "target_aug_auprc_std",
        ]

        out = out.merge(aug_out, on="base_model", how="left")

        # if verbose:
        #     print(f"[{task}] orig models:", sorted(work['base_model'].unique()))
        #     print(f"[{task}] aug  models:", sorted(aug_work['base_model'].unique()))

    out["display_name"] = out["base_model"].apply(get_display_name)
    out = reorder_methods(out)
    return out


def get_top2_mask(values: pd.Series):
    vals = values.astype(float).replace([np.inf, -np.inf], np.nan).dropna()
    if len(vals) == 0:
        return {}

    sorted_idx = vals.sort_values(ascending=False).index.tolist()
    marks = {}
    if len(sorted_idx) >= 1:
        marks[sorted_idx[0]] = "best"
    if len(sorted_idx) >= 2:
        marks[sorted_idx[1]] = "second"
    return marks


def latex_cell(mean_val, std_val, rank_label=None):
    if pd.isna(mean_val):
        return "-"
    core = f"\\valstd{{{mean_val * 100:.1f}}}{{{std_val * 100:.1f}}}"
    if rank_label == "best":
        core = f"\\valstdb{{{mean_val * 100:.1f}}}{{{std_val * 100:.1f}}}"
    elif rank_label == "second":
        core = f"\\valstdu{{{mean_val * 100:.1f}}}{{{std_val * 100:.1f}}}"
    return core


def make_latex_table(
    task_summary: pd.DataFrame,
    task: str,
    source_name: str,
    target_name: str,
    include_aug_target=False,
    aug_target_name="Target-Aug"
):
    pretty_task = pretty_task_name(task)

    cols = {
        "source_auroc": f"test/source_auroc_{task}_mean",
        "source_auprc": f"test/source_auprc_{task}_mean",
        "target_auroc": f"test/target_auroc_{task}_mean",
        "target_auprc": f"test/target_auprc_{task}_mean",
    }
    std_cols = {
        "source_auroc": f"test/source_auroc_{task}_std",
        "source_auprc": f"test/source_auprc_{task}_std",
        "target_auroc": f"test/target_auroc_{task}_std",
        "target_auprc": f"test/target_auprc_{task}_std",
    }

    if include_aug_target:
        cols["target_aug_auroc"] = "target_aug_auroc_mean"
        cols["target_aug_auprc"] = "target_aug_auprc_mean"
        std_cols["target_aug_auroc"] = "target_aug_auroc_std"
        std_cols["target_aug_auprc"] = "target_aug_auprc_std"

    rank_marks = {}
    for key, mean_col in cols.items():
        rank_marks[key] = get_top2_mask(task_summary[mean_col])

    lines = []
    lines.append("\\begin{table}[!t]")
    lines.append("\\scriptsize")
    lines.append("\\centering")
    lines.append(
        f"\\caption{{Task-wise zero-shot performance for \\texttt{{{pretty_task}}} "
        f"when training on {source_name} and evaluating on {source_name} (source), "
        f"{target_name} (target)"
        + (f", and {aug_target_name} (augmented target)" if include_aug_target else "")
        + ".}"
    )
    lines.append(f"\\label{{tab:{source_name.lower()}_{task.lower()}_task}}")
    lines.append("\\resizebox{\\textwidth}{!}{")

    if include_aug_target:
        lines.append("\\begin{tabular}{l|ll|ll|ll}")
        lines.append("\\toprule")
        lines.append(
            f"& \\multicolumn{{2}}{{c|}}{{Source ({source_name})}} "
            f"& \\multicolumn{{2}}{{c|}}{{Target ({target_name})}} "
            f"& \\multicolumn{{2}}{{c}}{{Target ({aug_target_name})}} \\\\"
        )
        lines.append("\\cmidrule{2-7}")
        lines.append("\\multirow{-2}{*}{Methods} & AUROC & AUPRC & AUROC & AUPRC & AUROC & AUPRC \\\\")
    else:
        lines.append("\\begin{tabular}{l|ll|ll}")
        lines.append("\\toprule")
        lines.append(
            f"& \\multicolumn{{2}}{{c|}}{{Source ({source_name})}} & "
            f"\\multicolumn{{2}}{{c}}{{Target ({target_name})}} \\\\"
        )
        lines.append("\\cmidrule{2-5}")
        lines.append("\\multirow{-2}{*}{Methods} & AUROC & AUPRC & AUROC & AUPRC \\\\")

    lines.append("\\midrule")

    for idx, row in task_summary.iterrows():
        m = row["display_name"]

        s_auroc = latex_cell(row[cols["source_auroc"]], std_cols["source_auroc"] in row and row[std_cols["source_auroc"]] or np.nan, rank_marks["source_auroc"].get(idx))
        s_auprc = latex_cell(row[cols["source_auprc"]], std_cols["source_auprc"] in row and row[std_cols["source_auprc"]] or np.nan, rank_marks["source_auprc"].get(idx))
        t_auroc = latex_cell(row[cols["target_auroc"]], std_cols["target_auroc"] in row and row[std_cols["target_auroc"]] or np.nan, rank_marks["target_auroc"].get(idx))
        t_auprc = latex_cell(row[cols["target_auprc"]], std_cols["target_auprc"] in row and row[std_cols["target_auprc"]] or np.nan, rank_marks["target_auprc"].get(idx))

        if include_aug_target:
            ta_auroc = latex_cell(row[cols["target_aug_auroc"]], row[std_cols["target_aug_auroc"]], rank_marks["target_aug_auroc"].get(idx))
            ta_auprc = latex_cell(row[cols["target_aug_auprc"]], row[std_cols["target_aug_auprc"]], rank_marks["target_aug_auprc"].get(idx))
            lines.append(f"{m} & {s_auroc} & {s_auprc} & {t_auroc} & {t_auprc} & {ta_auroc} & {ta_auprc}\\\\")
        else:
            lines.append(f"{m} & {s_auroc} & {s_auprc} & {t_auroc} & {t_auprc}\\\\")

    lines.append("\\bottomrule")
    lines.append("\\end{tabular}")
    lines.append("}")
    lines.append("\\end{table}")

    return "\n".join(lines)


def make_readable_table(task_summary: pd.DataFrame, task: str, include_aug_target=False):
    out = pd.DataFrame({
        "Methods": task_summary["display_name"],
        "Source AUROC": [
            fmt_mean_std(m, s) for m, s in zip(
                task_summary[f"test/source_auroc_{task}_mean"],
                task_summary[f"test/source_auroc_{task}_std"]
            )
        ],
        "Source AUPRC": [
            fmt_mean_std(m, s) for m, s in zip(
                task_summary[f"test/source_auprc_{task}_mean"],
                task_summary[f"test/source_auprc_{task}_std"]
            )
        ],
        "Target AUROC": [
            fmt_mean_std(m, s) for m, s in zip(
                task_summary[f"test/target_auroc_{task}_mean"],
                task_summary[f"test/target_auroc_{task}_std"]
            )
        ],
        "Target AUPRC": [
            fmt_mean_std(m, s) for m, s in zip(
                task_summary[f"test/target_auprc_{task}_mean"],
                task_summary[f"test/target_auprc_{task}_std"]
            )
        ],
    })

    if include_aug_target:
        out["Target Aug AUROC"] = [
            fmt_mean_std(m, s) for m, s in zip(
                task_summary["target_aug_auroc_mean"],
                task_summary["target_aug_auroc_std"]
            )
        ]
        out["Target Aug AUPRC"] = [
            fmt_mean_std(m, s) for m, s in zip(
                task_summary["target_aug_auprc_mean"],
                task_summary["target_aug_auprc_std"]
            )
        ]

    return out


def plot_task_performance(
    task_summary: pd.DataFrame,
    task: str,
    out_dir: str,
    include_aug_target=False,
    aug_target_name="Target-Aug"
):
    models = task_summary["display_name"].tolist()
    x = np.arange(len(models))

    if include_aug_target:
        width = 0.16
    else:
        width = 0.2

    s_auroc = task_summary[f"test/source_auroc_{task}_mean"].to_numpy() * 100
    s_auprc = task_summary[f"test/source_auprc_{task}_mean"].to_numpy() * 100
    t_auroc = task_summary[f"test/target_auroc_{task}_mean"].to_numpy() * 100
    t_auprc = task_summary[f"test/target_auprc_{task}_mean"].to_numpy() * 100

    s_auroc_std = task_summary[f"test/source_auroc_{task}_std"].fillna(0).to_numpy() * 100
    s_auprc_std = task_summary[f"test/source_auprc_{task}_std"].fillna(0).to_numpy() * 100
    t_auroc_std = task_summary[f"test/target_auroc_{task}_std"].fillna(0).to_numpy() * 100
    t_auprc_std = task_summary[f"test/target_auprc_{task}_std"].fillna(0).to_numpy() * 100

    fig, ax = plt.subplots(figsize=FIGSIZE)

    if include_aug_target:
        ta_auroc = task_summary["target_aug_auroc_mean"].to_numpy() * 100
        ta_auprc = task_summary["target_aug_auprc_mean"].to_numpy() * 100
        ta_auroc_std = task_summary["target_aug_auroc_std"].fillna(0).to_numpy() * 100
        ta_auprc_std = task_summary["target_aug_auprc_std"].fillna(0).to_numpy() * 100

        aug_palette = PALETTE + ["#B7B1D8", "#E7CFC4"]

        ax.bar(x - 2.5 * width, s_auroc, width, yerr=s_auroc_std, capsize=3, label=f"Source AUROC ({SOURCE_NAME})", color=aug_palette[0])
        ax.bar(x - 1.5 * width, s_auprc, width, yerr=s_auprc_std, capsize=3, label=f"Source AUPRC ({SOURCE_NAME})", color=aug_palette[1])
        ax.bar(x - 0.5 * width, t_auroc, width, yerr=t_auroc_std, capsize=3, label=f"Target AUROC ({TARGET_NAME})", color=aug_palette[2])
        ax.bar(x + 0.5 * width, t_auprc, width, yerr=t_auprc_std, capsize=3, label=f"Target AUPRC ({TARGET_NAME})", color=aug_palette[3])
        ax.bar(x + 1.5 * width, ta_auroc, width, yerr=ta_auroc_std, capsize=3, label=f"Target AUROC ({aug_target_name})", color=aug_palette[4])
        ax.bar(x + 2.5 * width, ta_auprc, width, yerr=ta_auprc_std, capsize=3, label=f"Target AUPRC ({aug_target_name})", color=aug_palette[5])
    else:
        ax.bar(x - 1.5 * width, s_auroc, width, yerr=s_auroc_std, capsize=3, label=f"Source AUROC ({SOURCE_NAME})", color=PALETTE[0])
        ax.bar(x - 0.5 * width, s_auprc, width, yerr=s_auprc_std, capsize=3, label=f"Source AUPRC ({SOURCE_NAME})", color=PALETTE[1])
        ax.bar(x + 0.5 * width, t_auroc, width, yerr=t_auroc_std, capsize=3, label=f"Target AUROC ({TARGET_NAME})", color=PALETTE[2])
        ax.bar(x + 1.5 * width, t_auprc, width, yerr=t_auprc_std, capsize=3, label=f"Target AUPRC ({TARGET_NAME})", color=PALETTE[3])

    ax.set_xticks(x)
    ax.set_xticklabels(models, rotation=35, ha="right")
    ax.set_ylabel("Score (%)")
    ax.set_title(f"Task-wise performance: {pretty_task_name(task)}")

    ax.legend(
    loc="upper center",
    bbox_to_anchor=(0.5, -0.18),
    ncol=3 if include_aug_target else 2,
    frameon=False
    )

    ax.grid(axis="y", linestyle="--", alpha=0.25)
    fig.tight_layout(rect=[0, 0.15, 1, 1])

    save_path = os.path.join(out_dir, f"{task}_performance.png")
    fig.savefig(save_path, dpi=DPI, bbox_inches="tight")
    plt.close(fig)

    return save_path
    

# ============================================================
# Average rank computation across tasks
# ============================================================
def compute_average_ranks(results_dict, tasks, include_aug_target=False):
    rank_rows = []

    metric_keys = [
        "source_auroc",
        "source_auprc",
        "target_auroc",
        "target_auprc",
    ]

    if include_aug_target:
        metric_keys += [
            "target_aug_auroc",
            "target_aug_auprc",
        ]

    for task in tasks:
        summary = results_dict[task]["summary_raw"].copy()

        for metric in metric_keys:
            if metric.startswith("target_aug"):
                mean_col = f"{metric}_mean"
            else:
                mean_col = f"test/{metric}_{task}_mean"

            if mean_col not in summary.columns:
                continue

            tmp = summary[["base_model", "display_name", mean_col]].copy()
            tmp = tmp.rename(columns={mean_col: "score"})
            tmp["rank"] = tmp["score"].rank(ascending=False, method="average")
            tmp["task"] = task
            tmp["metric"] = metric

            rank_rows.append(tmp[["task", "metric", "base_model", "display_name", "score", "rank"]])

    if len(rank_rows) == 0:
        raise ValueError("No rank rows computed. Check task/metric columns.")

    rank_detail = pd.concat(rank_rows, axis=0, ignore_index=True)

    rank_table = (
        rank_detail.groupby(["base_model", "display_name", "metric"], as_index=False)["rank"]
        .mean()
        .rename(columns={"rank": "avg_rank"})
    )

    metric_order_map = {
        "source_auroc": 0,
        "source_auprc": 1,
        "target_auroc": 2,
        "target_auprc": 3,
    }
    if include_aug_target:
        metric_order_map["target_aug_auroc"] = 4
        metric_order_map["target_aug_auprc"] = 5

    rank_table["metric_order"] = rank_table["metric"].map(metric_order_map)
    rank_table["model_order"] = rank_table["base_model"].apply(
        lambda x: METHOD_ORDER.index(x) if x in METHOD_ORDER else 999
    )

    rank_table = rank_table.sort_values(["model_order", "metric_order"]).drop(
        columns=["metric_order", "model_order"]
    )

    rank_table_wide = rank_table.pivot(
        index=["base_model", "display_name"],
        columns="metric",
        values="avg_rank"
    ).reset_index()

    desired_metric_order = ["source_auroc", "source_auprc", "target_auroc", "target_auprc"]
    if include_aug_target:
        desired_metric_order += ["target_aug_auroc", "target_aug_auprc"]

    for col in desired_metric_order:
        if col not in rank_table_wide.columns:
            rank_table_wide[col] = np.nan

    rank_table_wide = rank_table_wide[["base_model", "display_name"] + desired_metric_order].copy()
    rank_table_wide["overall_avg_rank"] = rank_table_wide[desired_metric_order].mean(axis=1)

    rank_table_wide["model_order"] = rank_table_wide["base_model"].apply(
        lambda x: METHOD_ORDER.index(x) if x in METHOD_ORDER else 999
    )
    rank_table_wide = rank_table_wide.sort_values("model_order").drop(columns="model_order").reset_index(drop=True)

    rename_map = {
        "display_name": "Methods",
        "source_auroc": "Source AUROC Avg Rank",
        "source_auprc": "Source AUPRC Avg Rank",
        "target_auroc": "Target AUROC Avg Rank",
        "target_auprc": "Target AUPRC Avg Rank",
        "overall_avg_rank": "Overall Avg Rank",
    }
    if include_aug_target:
        rename_map["target_aug_auroc"] = "Target Aug AUROC Avg Rank"
        rename_map["target_aug_auprc"] = "Target Aug AUPRC Avg Rank"

    rank_table_wide = rank_table_wide.rename(columns=rename_map)

    numeric_cols = [c for c in rank_table_wide.columns if c.endswith("Avg Rank")]
    rank_table_wide[numeric_cols] = rank_table_wide[numeric_cols].round(3)

    return rank_table_wide, rank_detail
    """
    For each metric:
      - rank models within each task (1 = best)
      - average ranks across tasks

    Returns:
      rank_table: wide table of average ranks
      rank_detail: long table with per-task ranks
    """
    rank_rows = []

    metric_keys = [
        "source_auroc",
        "source_auprc",
        "target_auroc",
        "target_auprc",
    ]

    for task in tasks:
        summary = results_dict[task]["summary_raw"].copy()

        for metric in metric_keys:
            mean_col = f"test/{metric}_{task}_mean"
            if mean_col not in summary.columns:
                continue

            tmp = summary[["base_model", "display_name", mean_col]].copy()
            tmp = tmp.rename(columns={mean_col: "score"})

            # higher is better, tied score gets same average rank
            tmp["rank"] = tmp["score"].rank(ascending=False, method="average")

            tmp["task"] = task
            tmp["metric"] = metric

            rank_rows.append(tmp[["task", "metric", "base_model", "display_name", "score", "rank"]])

    if len(rank_rows) == 0:
        raise ValueError("No rank rows computed. Check task/metric columns.")

    rank_detail = pd.concat(rank_rows, axis=0, ignore_index=True)

    rank_table = (
        rank_detail.groupby(["base_model", "display_name", "metric"], as_index=False)["rank"]
        .mean()
        .rename(columns={"rank": "avg_rank"})
    )

    rank_table["metric_order"] = rank_table["metric"].map({
        "source_auroc": 0,
        "source_auprc": 1,
        "target_auroc": 2,
        "target_auprc": 3,
    })
    rank_table["model_order"] = rank_table["base_model"].apply(
        lambda x: METHOD_ORDER.index(x) if x in METHOD_ORDER else 999
    )

    rank_table = rank_table.sort_values(["model_order", "metric_order"]).drop(
        columns=["metric_order", "model_order"]
    )

    rank_table_wide = rank_table.pivot(
        index=["base_model", "display_name"],
        columns="metric",
        values="avg_rank"
    ).reset_index()

    desired_metric_order = ["source_auroc", "source_auprc", "target_auroc", "target_auprc"]
    for col in desired_metric_order:
        if col not in rank_table_wide.columns:
            rank_table_wide[col] = np.nan

    rank_table_wide = rank_table_wide[
        ["base_model", "display_name"] + desired_metric_order
    ].copy()

    rank_table_wide["overall_avg_rank"] = rank_table_wide[desired_metric_order].mean(axis=1)

    rank_table_wide["model_order"] = rank_table_wide["base_model"].apply(
        lambda x: METHOD_ORDER.index(x) if x in METHOD_ORDER else 999
    )
    rank_table_wide = rank_table_wide.sort_values("model_order").drop(columns="model_order").reset_index(drop=True)

    rank_table_wide = rank_table_wide.rename(columns={
        "display_name": "Methods",
        "source_auroc": "Source AUROC Avg Rank",
        "source_auprc": "Source AUPRC Avg Rank",
        "target_auroc": "Target AUROC Avg Rank",
        "target_auprc": "Target AUPRC Avg Rank",
        "overall_avg_rank": "Overall Avg Rank",
    })

    numeric_cols = [
        "Source AUROC Avg Rank",
        "Source AUPRC Avg Rank",
        "Target AUROC Avg Rank",
        "Target AUPRC Avg Rank",
        "Overall Avg Rank",
    ]
    rank_table_wide[numeric_cols] = rank_table_wide[numeric_cols].round(3)

    return rank_table_wide, rank_detail


def plot_average_ranks(rank_table_wide: pd.DataFrame, out_dir: str):
    plot_df = rank_table_wide.copy()
    models = plot_df["Methods"].tolist()
    x = np.arange(len(models))

    metric_cols = [c for c in plot_df.columns if c.endswith("Avg Rank") and c != "Overall Avg Rank"]
    n_metrics = len(metric_cols)
    width = 0.8 / n_metrics

    soft_colors = ["#8FB9A8", "#F2C6A0", "#A8C3D1", "#D6A2AD", "#B7B1D8", "#E7CFC4"]

    fig, ax = plt.subplots(figsize=(12.5, 5.8))

    offsets = np.linspace(-(n_metrics - 1) / 2, (n_metrics - 1) / 2, n_metrics) * width

    for i, col in enumerate(metric_cols):
        ax.bar(
            x + offsets[i],
            plot_df[col].to_numpy(),
            width,
            label=col,
            color=soft_colors[i % len(soft_colors)]
        )

    ax.set_xticks(x)
    ax.set_xticklabels(models, rotation=35, ha="right")
    ax.set_ylabel("Average Rank (lower is better)")
    ax.set_title("Average model ranks across tasks")
    ax.invert_yaxis()

    ax.legend(
        loc="lower center",
        bbox_to_anchor=(0.5, 1.02),
        ncol=min(3, n_metrics),
        frameon=False,
        borderaxespad=0.0
    )

    ax.grid(axis="y", linestyle="--", alpha=0.25)
    fig.tight_layout(rect=[0, 0, 1, 0.90])

    save_path = os.path.join(out_dir, "average_ranks.png")
    fig.savefig(save_path, dpi=DPI, bbox_inches="tight")
    plt.close(fig)

    return save_path
# ============================================================
# Main driver
# ============================================================
def build_taskwise_results(
    csv_path: str,
    out_dir: str,
    source_name: str,
    target_name: str,
    tasks=None,
    include_aug_target=False,
    aug_csv_path=None,
    aug_target_name="Target-Aug",
):
    df = pd.read_csv(csv_path)
    aug_df = None

    if include_aug_target:
        if aug_csv_path is None:
            raise ValueError("include_aug_target=True but aug_csv_path is None")
        aug_df = pd.read_csv(aug_csv_path)

    all_tasks = extract_tasks(df)
    if tasks is None:
        tasks = all_tasks

    results = {}
    latex_tables = {}

    for task in tasks:
        print(f"[Processing] {task}")
        summary = summarize_task(df, task, aug_df=aug_df)
        readable = make_readable_table(summary, task, include_aug_target=include_aug_target)
        latex = make_latex_table(
            summary,
            task,
            source_name,
            target_name,
            include_aug_target=include_aug_target,
            aug_target_name=aug_target_name,
        )
        fig_path = plot_task_performance(
            summary,
            task,
            out_dir,
            include_aug_target=include_aug_target,
            aug_target_name=aug_target_name,
        )

        readable_csv_path = os.path.join(out_dir, f"{task}_summary.csv")
        latex_path = os.path.join(out_dir, f"{task}_table.tex")

        readable.to_csv(readable_csv_path, index=False)
        with open(latex_path, "w", encoding="utf-8") as f:
            f.write(latex)

        results[task] = {
            "summary_raw": summary,
            "summary_readable": readable,
            "plot_path": fig_path,
            "latex_path": latex_path,
            "csv_path": readable_csv_path,
        }
        latex_tables[task] = latex

    # ------------------------------------------------------------
    # phenotype summary:
    # first macro over phenotype tasks within each run,
    # then average over runs
    # ------------------------------------------------------------
    phenotype_summary, phenotype_tasks, phenotype_run_macro = summarize_phenotype_from_raw(
        df,
        aug_df=aug_df,
        exclude_tasks={"readm_label", "los_label", "mor_label"},
        verbose=False,
    )

    phenotype_readable = make_readable_table_phenotype(
        phenotype_summary,
        include_aug_target=include_aug_target
    )

    phenotype_plot = plot_phenotype_performance(
        phenotype_summary,
        out_dir,
        include_aug_target=include_aug_target,
        aug_target_name=aug_target_name,
    )
    phenotype_latex = make_latex_table_phenotype(
        phenotype_summary,
        source_name=source_name,
        target_name=target_name,
        include_aug_target=include_aug_target,
        aug_target_name=aug_target_name,
    )



    phenotype_csv_path = os.path.join(out_dir, "phenotype_summary.csv")
    phenotype_runs_csv_path = os.path.join(out_dir, "phenotype_run_macro.csv")

    phenotype_readable.to_csv(phenotype_csv_path, index=False)
    phenotype_run_macro.to_csv(phenotype_runs_csv_path, index=False)

    phenotype_latex_path = os.path.join(out_dir, "phenotype_table.tex")
    with open(phenotype_latex_path, "w", encoding="utf-8") as f:
        f.write(phenotype_latex)

    results["phenotype"] = {
        "summary_raw": phenotype_summary,
        "summary_readable": phenotype_readable,
        "plot_path": phenotype_plot,
        "csv_path": phenotype_csv_path,
        "latex_path": phenotype_latex_path,
        "run_macro_csv_path": phenotype_runs_csv_path,
        "phenotype_tasks_used": phenotype_tasks,
    }
    latex_tables["phenotype"] = phenotype_latex
    rank_tasks = [t for t in tasks if t in results]  # phenotype 제외
    avg_rank_table, rank_detail = compute_average_ranks(
        results,
        rank_tasks,
        include_aug_target=include_aug_target
    )

    avg_rank_csv = os.path.join(out_dir, "average_ranks.csv")
    rank_detail_csv = os.path.join(out_dir, "rank_detail_per_task.csv")
    avg_rank_plot = plot_average_ranks(avg_rank_table, out_dir)

    avg_rank_table.to_csv(avg_rank_csv, index=False)
    rank_detail.to_csv(rank_detail_csv, index=False)

    return results, latex_tables, avg_rank_table, rank_detail, avg_rank_plot

# ============================================================
# Run
# ============================================================
results, latex_tables, avg_rank_table, rank_detail, avg_rank_plot = build_taskwise_results(
    csv_path=CSV_PATH,
    out_dir=OUT_DIR,
    source_name=SOURCE_NAME,
    target_name=TARGET_NAME,
    include_aug_target=INCLUDE_AUG_TARGET,
    aug_csv_path=AUG_CSV_PATH,
    aug_target_name=AUG_TARGET_NAME,
)

print("\nDone.")
print(f"Saved outputs under: {OUT_DIR}")

print("\nAverage rank table:")
print(avg_rank_table)

print(f"\nAverage rank plot saved to: {avg_rank_plot}")

# Example: inspect one task
example_task = list(results.keys())[0]
print(f"\nExample task: {example_task}")
print(results[example_task]["summary_readable"])

[Processing] Acute_and_unspecified_renal_failure
[Processing] Acute_cerebrovascular_disease
[Processing] Acute_myocardial_infarction
[Processing] Cardiac_dysrhythmias
[Processing] Chronic_kidney_disease
[Processing] Chronic_obstructive_pulmonary_disease_and_bronchiectasis
[Processing] Complications_of_surgical_procedures_or_medical_care
[Processing] Conduction_disorders
[Processing] Congestive_heart_failure;_nonhypertensive
[Processing] Coronary_atherosclerosis_and_other_heart_disease
[Processing] Diabetes_mellitus_with_complications
[Processing] Diabetes_mellitus_without_complication
[Processing] Disorders_of_lipid_metabolism
[Processing] Essential_hypertension
[Processing] Fluid_and_electrolyte_disorders
[Processing] Gastrointestinal_hemorrhage
[Processing] Hypertension_with_complications_and_secondary_hypertension
[Processing] Other_liver_diseases
[Processing] Other_lower_respiratory_disease
[Processing] Other_upper_respiratory_disease
[Processing] Pleurisy;_pneumothorax;_pulmonary_

In [11]:
# ============================================================
# Config
# ============================================================
CSV_PATH = "eicu_results.csv"   # <- change if needed
OUT_DIR = "taskwise_results_eicu"
os.makedirs(OUT_DIR, exist_ok=True)
AUG_CSV_PATH = "mimic_aug_results.csv"   # optional
INCLUDE_AUG_TARGET = False
AUG_TARGET_NAME = "eICU-Augmented"

SOURCE_NAME = "eICU"   # <- change to "eICU" if this csv is eICU-source experiment
TARGET_NAME = "MIMIC"    # <- change accordingly

# plotting: larger is fine, don't manually set colors per instruction
FIGSIZE = (11, 5)
DPI = 180

# ============================================================
# Run
# ============================================================
results, latex_tables, avg_rank_table, rank_detail, avg_rank_plot = build_taskwise_results(
    csv_path=CSV_PATH,
    out_dir=OUT_DIR,
    source_name=SOURCE_NAME,
    target_name=TARGET_NAME,
    include_aug_target=INCLUDE_AUG_TARGET,
    aug_csv_path=AUG_CSV_PATH,
    aug_target_name=AUG_TARGET_NAME,
)

print("\nDone.")
print(f"Saved outputs under: {OUT_DIR}")

print("\nAverage rank table:")
print(avg_rank_table)

print(f"\nAverage rank plot saved to: {avg_rank_plot}")

# Example: inspect one task
example_task = list(results.keys())[0]
print(f"\nExample task: {example_task}")
print(results[example_task]["summary_readable"])

[Processing] Acute_and_unspecified_renal_failure
[Processing] Acute_cerebrovascular_disease
[Processing] Acute_myocardial_infarction
[Processing] Cardiac_dysrhythmias
[Processing] Chronic_kidney_disease
[Processing] Chronic_obstructive_pulmonary_disease_and_bronchiectasis
[Processing] Complications_of_surgical_procedures_or_medical_care
[Processing] Conduction_disorders
[Processing] Congestive_heart_failure;_nonhypertensive
[Processing] Coronary_atherosclerosis_and_other_heart_disease
[Processing] Diabetes_mellitus_with_complications
[Processing] Diabetes_mellitus_without_complication
[Processing] Disorders_of_lipid_metabolism
[Processing] Essential_hypertension
[Processing] Fluid_and_electrolyte_disorders
[Processing] Gastrointestinal_hemorrhage
[Processing] Hypertension_with_complications_and_secondary_hypertension
[Processing] Other_liver_diseases
[Processing] Other_lower_respiratory_disease
[Processing] Other_upper_respiratory_disease
[Processing] Pleurisy;_pneumothorax;_pulmonary_